In [1]:
import json
import pandas as pd

valid_records = []
invalid_record_count = 0

with open("results/20260729_093018.records.jsonl", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
            
        try:
            valid_records.append(json.loads(line))
        except json.JSONDecodeError:
            invalid_record_count += 1
            # Skip the corrupted line (usually the very last one where you killed the script)
            pass

# pd.json_normalize automatically flattens nested dictionaries.
# A JSON structure like {"result": {"score": 0.8}} becomes a column named "result.score"
df = pd.json_normalize(valid_records)

print(f"Loaded {len(df)}/{len(df) + invalid_record_count} valid records.")

Loaded 653/653 valid records.


In [2]:
df.columns

Index(['setting.prompt.setting.system_prompt',
       'setting.prompt.setting.response_format.type',
       'setting.prompt.setting.prompt_format', 'setting.prompt.name',
       'setting.model', 'input.text', 'input.language.source_language',
       'input.language.target_language', 'result.translation', 'result.rating',
       'metadata.created_at', 'metadata.success', 'metadata.error',
       'metadata.elapsed_seconds'],
      dtype='str')

In [3]:
df['setting.model']

0      qwen2.5-1.5b
1      qwen2.5-1.5b
2      qwen2.5-1.5b
3      smollm2-1.7b
4      smollm2-1.7b
           ...     
648    qwen2.5-1.5b
649    qwen2.5-1.5b
650    qwen2.5-1.5b
651    smollm2-1.7b
652    smollm2-1.7b
Name: setting.model, Length: 653, dtype: str

In [4]:
all(df[df['setting.model'] == 'qwen2.5-1.5b']['setting.model'] == 'qwen2.5-1.5b')

True

In [5]:

pd.set_option('display.max_colwidth', 60)
df[["input.text", "result.translation", "result.rating", "setting.model", "setting.prompt.name"]].sort_values("result.rating")

,input.text,result.translation,result.rating,setting.model,setting.prompt.name
262,"by those who take the lead, racing,","berkomunikasi, berkomunikasi, berkomunikasi, berkomunika...",-1.973964,smollm2-1.7b,professional
67,and build above you the seven mighty heavens?,dan bangun di atasmu tujuh heksaheksa heksaheksa heksahe...,-1.962424,qwen2.5-1.5b,professional
406,"He raised its vault, and fashioned it,","Dia mengambil tumpuan, dan duniaannya, dan duniaannya du...",-1.941548,smollm2-1.7b,professional
238,Indeed We have warned you\n of a punishment near at h...,Saya telah membawa kekas-kesan kepada kamu\n bukan ke...,-1.687151,smollm2-1.7b,professional
71,and build above you the seven mighty heavens?,berkatannya di atas kamu dua dua kubu tiga dua dua empat...,-1.541120,smollm2-1.7b,verbose
...,...,...,...,...,...
322,"and behold, they will be awake.","and behold, they will be awake.",0.661248,smollm2-1.7b,professional
580,No indeed! These [verses of the Qur'ān] are a reminder,No indeed! These [verses of the Qur'ān] are a reminder.,0.695094,smollm2-1.7b,professional
20,No indeed! They will soon know!,"Tidak, ya! Mereka akan segera tahu!",0.752566,qwen2.5-1.5b,verbose
19,No indeed! They will soon know!,"Tidak, ya! Mereka akan segera tahu!",0.752566,qwen2.5-1.5b,professional


In [6]:
df.groupby(['setting.prompt.name', 'setting.model'])['metadata.elapsed_seconds'].describe()

count      mean       std       min  \
setting.prompt.name setting.model                                        
minimal             qwen2.5-1.5b   109.0  2.688333  0.525286  2.007235   
                    smollm2-1.7b   109.0  3.660433  5.169562  2.507353   
professional        qwen2.5-1.5b   109.0  2.981213  3.214719  2.007375   
                    smollm2-1.7b   109.0  4.532217  7.658730  2.507183   
verbose             qwen2.5-1.5b   109.0  3.105195  0.387970  2.507326   
                    smollm2-1.7b   108.0  3.972013  1.529018  3.007920   

                                        25%       50%       75%        max  
setting.prompt.name setting.model                                           
minimal             qwen2.5-1.5b   2.507648  2.508027  2.510005   6.515373  
                    smollm2-1.7b   2.508088  3.007853  3.009801  43.028048  
professional        qwen2.5-1.5b   2.507777  2.508261  3.007729  36.026252  
                    smollm2-1.7b   2.507860  3.007755  3.009396  40.525344  
verbose             qwen2.5-1.5b   3.007939  3.008339  3.009804   5.010861  
                    smollm2-1.7b   3.508071  3.509211  4.009056  14.517365

In [23]:
val = df[(df['setting.prompt.name'] == 'minimal') & (df['setting.model'] == 'qwen2.5-1.5b')]['input.text'].values
val

[v for v in val if any(k in v for k in '<{[(')]

['[Is it] about the great tiding,',
 'and create you in pairs?<{["F", 1]}>',
 'and make [the sun for] a radiant lamp?',
 'So [now] taste!\n\xa0We shall increase you in nothing but punishment!',
 '—a reward from _your_ Lord,\n\xa0\xa0\xa0a bounty sufficing,<{["F", 1]}>',
 'Indeed We have warned you\n\xa0\xa0\xa0\xa0of a punishment near at hand\n\xa0\xa0\xa0—the day when a person will observe<{["F", 1]}>\n\xa0\xa0\xa0\xa0what his hands have sent ahead\n\xa0\xa0\xa0and the faithless one will say,\n\xa0\xa0\xa0\'I wish I were dust!\'',
 'By those [angels] who wrest [the soul] violently,',
 'by those who draw [it] out gently,',
 'by those who direct the affairs [of creatures]:',
 'and is followed by the Successor,<{["F", 1]}>',
 "[And said,] 'Go to Pharaoh,\n\xa0\xa0for indeed he has rebelled,",
 'I will guide you to your Lord,\n\xa0\xa0\xa0that you may fear [Him]?" \'',
 'and mustered [the people] and proclaimed,',
 'as a [place of] sustenance for you\n\xa0\xa0\xa0and your livestock.',
 'B

In [54]:
def check_citations_intact(source: str, translation: str) -> bool:
    import re
    pattern = re.compile(r'<\{\["([\w]+)"\s*,\s*("[\w:]+"|\d+)\]\}>')
    return pattern.findall(source) == pattern.findall(translation)

In [56]:
print(check_citations_intact('See <{["Q", "112:3"]}>', 'Lihat <{["Q", "112:3"]}>'))
print(check_citations_intact('and is followed by the Successor,<{["F", 1]}>', 'dan diikuti Penerusnya,<{["F", 1]}>'))

True
True


In [83]:
intact_citations = df.apply(lambda row: check_citations_intact(row['input.text'], row['result.translation']), axis=1)
display(intact_citations.describe())
df[~intact_citations].groupby(['setting.prompt.name', 'setting.model'])[
    ['setting.prompt.name', 'setting.model', 'input.text', 'result.translation']
].apply(display)

count      653
unique       2
top       True
freq       634
dtype: object

,setting.prompt.name,setting.model,input.text,result.translation
42,minimal,qwen2.5-1.5b,"and create you in pairs?<{[""F"", 1]}>",English: And create you in pairs?\nIndonesian: Dan buatl...
234,minimal,qwen2.5-1.5b,Indeed We have warned you\n of a punishment near at h...,Tentu kita telah mengingatkan Anda\n akan hukuman yan...
276,minimal,qwen2.5-1.5b,"and is followed by the Successor,<{[""F"", 1]}>","English: And is followed by the Successor, {[""F"", 1]}.\n..."


,setting.prompt.name,setting.model,input.text,result.translation
45,minimal,smollm2-1.7b,"and create you in pairs?<{[""F"", 1]}>","{[""F"", 1]}"
213,minimal,smollm2-1.7b,"—a reward from _your_ Lord,\n a bounty sufficing,<{[""F...","—a reward from your Lord,\n a bounty sufficing,"
237,minimal,smollm2-1.7b,Indeed We have warned you\n of a punishment near at h...,Saya berkata kepada kamu bahwa kami telah berkata kepada...
279,minimal,smollm2-1.7b,"and is followed by the Successor,<{[""F"", 1]}>","dalam bahasa Inggris ke bahasa Indonesia, adalah ""beriku..."


,setting.prompt.name,setting.model,input.text,result.translation
235,professional,qwen2.5-1.5b,Indeed We have warned you\n of a punishment near at h...,"Tentu saja, kita telah mengingatkan Anda tentang hukuman..."
277,professional,qwen2.5-1.5b,"and is followed by the Successor,<{[""F"", 1]}>","dan diikuti oleh Successor, [{[""F"", 1]}]"


,setting.prompt.name,setting.model,input.text,result.translation
46,professional,smollm2-1.7b,"and create you in pairs?<{[""F"", 1]}>",dan membuat kamu dalam kumpulan
238,professional,smollm2-1.7b,Indeed We have warned you\n of a punishment near at h...,Saya telah membawa kekas-kesan kepada kamu\n bukan ke...
280,professional,smollm2-1.7b,"and is followed by the Successor,<{[""F"", 1]}>","dan berlalu selepasnya,<[""F"", 1]>"
604,professional,smollm2-1.7b,"in the hands of envoys,<{[""F"", 1]}>","di tangga-tangga wakil-wakil,"


,setting.prompt.name,setting.model,input.text,result.translation
212,verbose,qwen2.5-1.5b,"—a reward from _your_ Lord,\n a bounty sufficing,<{[""F...","—pemberian dari _Raja_ Anda,\n denda yang cukup,<{[""F""..."
236,verbose,qwen2.5-1.5b,Indeed We have warned you\n of a punishment near at h...,"Tentu saja, kita telah mengingatkan Anda tentang hukuman..."
278,verbose,qwen2.5-1.5b,"and is followed by the Successor,<{[""F"", 1]}>","dan diikuti oleh Successor, {[""F"", 1]}"
602,verbose,qwen2.5-1.5b,"in the hands of envoys,<{[""F"", 1]}>","dalam tangan diplomat, {[""F"", 1]}"


,setting.prompt.name,setting.model,input.text,result.translation
239,verbose,smollm2-1.7b,Indeed We have warned you\n of a punishment near at h...,Kami telah berkomentar kepadamu tentang penjagaan yang a...
605,verbose,smollm2-1.7b,"in the hands of envoys,<{[""F"", 1]}>","dalam tanggung jawab pengawas,"


,setting.prompt.name,setting.model,input.text,result.translation
